# Buscaminas MARIE - Notebook autosuficiente

Este notebook:

1. genera un tablero de Buscaminas 16×16,
2. genera una **máscara de forma** (`SHAPE_PRESET`) para cambiar la forma del tablero,
3. crea los archivos:
   - `marie/generados/buscaminas_generado_mod.mas`
   - `marie/generados/tablero_marie_generado.mem`
   - `marie/generados/shape_preset_generado.mem`
   - `outputs/vista_tablero_buscaminas.png`
4. deja todo listo para abrir el `.mas` en **Marie.js**.

Está pensado para funcionar tanto en **Google Colab** como en **local**.


In [ ]:
import os
import random
import requests
from pathlib import Path
from typing import List, Tuple, Optional

try:
    import matplotlib.pyplot as plt
    from matplotlib.colors import ListedColormap
except ImportError:
    !pip install -q matplotlib
    import matplotlib.pyplot as plt
    from matplotlib.colors import ListedColormap

# Crear estructura de carpetas
Path("marie/plantillas").mkdir(parents=True, exist_ok=True)
Path("marie/generados").mkdir(parents=True, exist_ok=True)
Path("outputs").mkdir(parents=True, exist_ok=True)

print("✅ Entorno preparado.")

## Parámetros del tablero y de la forma
Modifica esta celda para cambiar minas, semilla y forma.

In [ ]:
# Configuración del tablero
FILAS = 16
COLUMNAS = 16
MINAS = 25
SEMILLA = 42 # Cambia esto para obtener tableros diferentes

# Forma del tablero: "llena", "rombo", "cruz", "marco"
FORMA = "llena"

# URL de respaldo para la plantilla (Ajusta con tu usuario de GitHub)
URL_PLANTILLA_RAW = "https://raw.githubusercontent.com/maateonicolas/buscaminas-marie/main/marie/plantillas/buscaminas_template_mod.mas"

In [ ]:
def generar_tablero_completo(f, c, n_minas, semilla):
    random.seed(semilla)
    tablero = [[0 for _ in range(c)] for _ in range(f)]

    # Colocar minas
    colocadas = 0
    while colocadas < n_minas:
        rf, rc = random.randint(0, f-1), random.randint(0, c-1)
        if tablero[rf][rc] != -1:
            tablero[rf][rc] = -1
            colocadas += 1

    # Calcular adyacencias
    for i in range(f):
        for j in range(c):
            if tablero[i][j] == -1: continue
            minas_cerca = 0
            for df in [-1,0,1]:
                for dc in [-1,0,1]:
                    if 0 <= i+df < f and 0 <= j+dc < c:
                        if tablero[i+df][j+dc] == -1: minas_cerca += 1
            tablero[i][j] = minas_cerca
    return tablero

def calcular_mapa_regiones(tablero, f, c):
    """
    Calcula grupos de celdas que deben revelarse juntas.
    ID 0 significa que no pertenece a ninguna región de expansión (es una mina o número aislado).
    """
    region_map = [[0 for _ in range(c)] for _ in range(f)]
    visitado = set()
    current_id = 1

    for i in range(f):
        for j in range(c):
            if tablero[i][j] == 0 and (i, j) not in visitado:
                cola = [(i, j)]
                visitado.add((i, j))
                while cola:
                    curr_f, curr_c = cola.pop(0)
                    region_map[curr_f][curr_c] = current_id
                    for df in [-1,0,1]:
                        for dc in [-1,0,1]:
                            nf, nc = curr_f+df, curr_c+dc
                            if 0 <= nf < f and 0 <= nc < c:
                                if tablero[nf][nc] == 0 and (nf, nc) not in visitado:
                                    visitado.add((nf, nc))
                                    cola.append((nf, nc))
                                elif tablero[nf][nc] > 0:
                                    region_map[nf][nc] = current_id
                current_id += 1
    return region_map

def matriz_a_marie_dec(matriz):
    lineal = [val for fila in matriz for val in fila]
    return "\n".join([f"    DEC {v}" for v in lineal])

print("✅ Funciones de lógica definidas.")

In [ ]:
import os
import requests
from pathlib import Path

# Configuración de rutas
REPO_URL = "https://raw.githubusercontent.com/maateonicolas/buscaminas-marie/main/marie/plantillas/buscaminas_template_mod.mas"
PATH_LOCAL = Path("marie/plantillas/buscaminas_template_mod.mas")

def cargar_plantilla():
    # 1. Intentar cargar desde la ruta local (Funciona en ejecución local de PyCharm/Jupyter)
    if PATH_LOCAL.exists():
        print(f"Cargando plantilla desde archivo local: {PATH_LOCAL}")
        return PATH_LOCAL.read_text(encoding="utf-8")

    # 2. Si no existe, intentar descargarla (Funciona en Google Colab)
    print("Archivo local no encontrado. Intentando descargar desde GitHub...")
    try:
        response = requests.get(REPO_URL)
        if response.status_code == 200:
            # Opcional: Guardar el archivo descargado para mantener la estructura de carpetas
            PATH_LOCAL.parent.mkdir(parents=True, exist_ok=True)
            PATH_LOCAL.write_text(response.text, encoding="utf-8")
            print("Plantilla descargada y guardada exitosamente.")
            return response.text
        else:
            raise Exception(f"Error al descargar: Código {response.status_code}")
    except Exception as e:
        print(f"No se pudo obtener la plantilla: {e}")
        # 3. Fallback de emergencia (Opcional: Una versión mínima por si falla internet)
        return "; Error: No se pudo cargar la lógica de MARIE"

template_mas = cargar_plantilla()

In [ ]:
# Generar datos
tablero = generar_tablero_completo(FILAS, COLUMNAS, MINAS, SEMILLA)
regiones = calcular_mapa_regiones(tablero, FILAS, COLUMNAS)
shape = [[1 for _ in range(COLUMNAS)] for _ in range(FILAS)] # Tablero lleno

# Visualización
plt.figure(figsize=(6,6))
plt.imshow(tablero, cmap='tab20')
for i in range(FILAS):
    for j in range(COLUMNAS):
        txt = "*" if tablero[i][j] == -1 else str(tablero[i][j])
        plt.text(j, i, txt, ha="center", va="center", color="white" if tablero[i][j] == -1 else "black")
plt.title(f"Vista Previa: {MINAS} Minas")
plt.savefig("outputs/vista_tablero_buscaminas.png")
plt.show()

In [ ]:
def exportar_proyecto(template, tablero, shape, regiones, destino):
    content = template.replace("__TABLERO_REAL__", matriz_a_marie_dec(tablero))
    content = content.replace("__SHAPE_PRESET__", matriz_a_marie_dec(shape))
    content = content.replace("__REGION_MAP__", matriz_a_marie_dec(regiones))

    with open(destino, "w") as f:
        f.write(content)
    print(f"🚀 Archivo generado: {destino}")

exportar_proyecto(
    template_mas,
    tablero,
    shape,
    regiones,
    "marie/generados/buscaminas_generado_mod.mas"
)